# Customer Churn Prediction

The Customer Churn table contains information on all 7,043 customers from a Telecommunications company in California in Q2 2022


Each record represents one customer, and contains details about their demographics, location, tenure, subscription services, status for the quarter (joined, stayed, or churned), and more!

The Zip Code Population table contains complimentary information on the estimated populations for the California zip codes in the Customer Churn table

Importing necessary files

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('../Datasets/telecom_customer_churn.csv')

In [ ]:
df.head(5)

Overviewing all the columns in the dataset

In [ ]:
df.columns

In [ ]:
df1 = df.copy()

Creating a copy of the Dataset

In [ ]:
df1.head(7)

In [ ]:
df1.columns

# Exploratory Data Analysis

## Data Preprocessing

Dropping unwanted columns from the dataset

In [ ]:
df1.drop(['Customer ID','Total Refunds','Zip Code','Latitude', 'Longitude','Churn Category', 'Churn Reason'],axis='columns',inplace=True)


In [ ]:
df1.shape

In [ ]:
df1.dtypes

Checking the number of unique values in each column

In [ ]:
features = df1.columns
for feature in features:
     print(f'{feature}--->{df[feature].nunique()}')

Getting the percentge of Null Values in each Column

In [ ]:
df1.isnull().sum() / df1.shape[0]

Cleaning Function for the Dataset

In [ ]:
def clean_dataset(df):
    assert isinstance(df, pd.DataFrame)
    df.dropna(inplace=True)
    indices_to_keep = ~df.isin([np.nan, np.inf, -np.inf]).any(1)
    return df[indices_to_keep].astype(np.float64)

In [ ]:
df1.dtypes

In [ ]:
df1.isnull().sum()

In [ ]:
# Fill numeric columns
num_cols = df1.select_dtypes(include=['int64', 'float64']).columns
df1[num_cols] = df1[num_cols].fillna(df1[num_cols].median())

# Fill categorical columns
cat_cols = df1.select_dtypes(include=['object', 'string']).columns
for col in cat_cols:
    df1[col] = df1[col].fillna(df1[col].mode()[0])

In [ ]:
df1=df1.dropna()
df.head()

In [ ]:
df['Unlimited Data'] 

In [ ]:
number_columns=['Age','Number of Dependents','Number of Referrals','Tenure in Months','Avg Monthly Long Distance Charges','Avg Monthly GB Download','Monthly Charge','Total Charges','Total Extra Data Charges','Total Long Distance Charges','Total Revenue']

Checking the unique values of column having datatype: 'object'

In [ ]:
def unique_values_names(df):
    for column in df:
        if df[column].dtype=='object':
            print(f'{column}:{df[column].unique()}')

In [ ]:
unique_values_names(df1)

# Data Visualization

In [ ]:
import plotly.express as px 

Visualizing Column 'Age' in the dataset

In [ ]:
fig = px.histogram(df1, x = 'Age')
fig.show()

Checking the stats in number_columns of the copied dataset

In [ ]:
df1.hist(figsize=(15,15), xrot=30)

In [ ]:
df1['Age']


In [ ]:
import matplotlib.pyplot as plt

Visualizing the number of customers who churned, stayed or joined in the company with a bar plot

In [ ]:
Customer_Stayed=df1[df1['Customer Status']=='Stayed'].Age
Customer_Churned=df1[df1['Customer Status']=='Churned'].Age
Customer_Joined=df1[df1['Customer Status']=='Joined'].Age

plt.xlabel('Age')
plt.ylabel('Customers Numbers')
plt.hist([Customer_Stayed,Customer_Churned,Customer_Joined], color=['black','red','blue'],label=['Stayed','Churned','Joined'])

plt.title('Customers Behavior ',fontweight ="bold")
plt.legend()

In [ ]:
import seaborn as sns

 ## Defining Correlation between the columns in the dataset

In [ ]:
data = df1.select_dtypes(include=['int64', 'float64']).corr()

plt.figure(figsize=(20,10))
sns.heatmap(data, annot=True)
plt.show()

Analyzing Outlier in the dataset with respect to customer status

In [ ]:
fig, ax = plt.subplots(4,3, figsize = (15,15))
for i, subplot in zip(number_columns, ax.flatten()):
    sns.boxplot(x = 'Customer Status', y = i , data = df1, ax = subplot)

In [ ]:
fig = px.density_heatmap(df1, x='Age', y='Total Charges')
fig.show()

In [ ]:
df1.columns

In [ ]:
pd.crosstab(df['Customer Status'], df['Married']).plot(kind='bar')

In [ ]:
pd.crosstab(df['Customer Status'], df['Gender']).plot(kind='bar')

In [ ]:
df1['Payment Method'].unique()

Create dictionary with role / data key value pairs

In [ ]:
Roles = {}
for j in df1['Payment Method'].unique():
    Roles[j] = df1[df1['Payment Method'] == j]

In [ ]:
Roles.keys()

Selecting the rows where the role is 'Credit Card'

In [ ]:
Roles['Credit Card']

In [ ]:
len(Roles)


Checking the number of Offers in the dataset

In [ ]:
off = df1['Offer'].value_counts()
off


In [ ]:
import plotly.graph_objects as go

In [ ]:
fig = go.Figure([go.Bar(x=off.index, y=off.values)])
fig.show()

In [ ]:
df1_off = Roles['Credit Card'].Offer.value_counts()
df1_off

In [ ]:
fig = go.Figure([go.Bar(x= df1_off.index, y=df1_off.values)])
fig.show()

In [ ]:
df1 = df1.rename(columns = {'Customer Status':'Customer_Status'})

In [ ]:
Roles1 = {}
for k in df1['Customer_Status'].unique():
    Roles1[k] = df1[df1['Customer_Status'] == k]
Roles1.keys()

In [ ]:
df1_state = Roles1['Stayed'].Offer.value_counts()
df1_state

# Data Modelling

Replacing the Gender column in the dataset with Label Encoding

0 for Female

1 for Male

In [ ]:
df1.replace({"Gender":{'Female':0,'Male':1}},inplace=True)

Replacing the columns with 'yes' and 'no' output by Label Encoding

0 for No

1 for Yes

In [ ]:
yes_and_no=[  'Paperless Billing', 'Unlimited Data', 
       'Streaming Movies', 'Streaming Music',  'Streaming TV',
       'Premium Tech Support', 'Device Protection Plan', 'Online Backup', 'Online Security',
       'Multiple Lines',  'Married']
for i in yes_and_no:
    df1.replace({'No':0,'Yes':1},inplace=True)

Replacing 'Phone Service' with '1'

In [ ]:
df1.replace({"Phone Service":{'Yes':1}},inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df1.Customer_Status = le.fit_transform(df1.Customer_Status)


In [ ]:
df1 = pd.get_dummies(data=df1, columns=['Payment Method','Contract','Internet Type','Offer','City'])


In [ ]:
cols_to_scale = ['Age','Number of Dependents','Number of Referrals','Tenure in Months','Avg Monthly Long Distance Charges','Avg Monthly GB Download','Monthly Charge', 'Total Charges',
       'Total Extra Data Charges', 'Total Long Distance Charges','Total Revenue']

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df1[cols_to_scale] = scaler.fit_transform(df1[cols_to_scale])

## Dealing with Imbalance Data

Dropping the Customer_Status

i.e. The column tht we have to predict and set as a dependent variable

In [ ]:
X = df1.drop('Customer_Status',axis='columns')
y = df1['Customer_Status']

In [ ]:
X.head(5)

In [ ]:
y.head(5)


# Data Model Building

## Spliiting the data in Training and Test Data

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=5)

In [ ]:
len(X_train)

In [ ]:
X_train[:10]

Importing the required files for the model that is to applied

1. Random Forest Classifier
2. Logistic Regression
3. GaussianNB
4. Decision Tree Classifier
5. XGB Classifier


## Importing Models

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

In [ ]:
data = df1.select_dtypes(include=['int64', 'float64']).corr()

plt.figure(figsize=(20,10))
sns.heatmap(data, annot=True)
plt.show()

In [ ]:
from sklearn.model_selection import ShuffleSplit

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

model_params = {
    'random_forest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [1, 5, 10]
        }
    },
    'logistic_regression': {
    'model': LogisticRegression(solver='lbfgs', max_iter=1000),
    'params': {
        'C': [1,5,10]
    }
    },
    'naive_bayes': {
        'model': GaussianNB(),
        'params': {}
    },
    'decision_tree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy']
        }
    }
}

In [ ]:
print("model_params" in globals())

Getting the best_score from the applied models

In [ ]:
from sklearn.model_selection import GridSearchCV
scores = []
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)
for model_name, mp in model_params.items():
    clf =  GridSearchCV(mp['model'], mp['params'], cv=cv, return_train_score=False)
    clf.fit(X,y)
    scores.append({
        'model': model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })
    
df = pd.DataFrame(scores,columns=['model','best_score','best_params'])
df

It was concluded that XGB_Classifier was giving us the best_score in the dataset

## Selecting the model with best score for the dataset

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df1.copy()

le = LabelEncoder()

for col in df_encoded.select_dtypes(include=['object']).columns:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

df_encoded.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X.dtypes

In [ ]:
X_train.dtypes[X_train.dtypes == 'object']

We got an accuracy of 80.86 percent in the testing dataset

## Predicting values from the model build to check the accuracy

In [ ]:
print(X_train.dtypes.value_counts()) 
X_train.select_dtypes(include=['object']).columns

## Verifying the actual values with the predicted values

In [ ]:
y_test[:5]

## Importing Confusion Matrx 

In [ ]:
X_train.select_dtypes(include=['object']).columns

In [ ]:
# Convert all categorical columns into numeric columns
df_encoded = pd.get_dummies(df1, drop_first=True)

print(df_encoded.dtypes.unique())

In [ ]:
print(df_encoded.columns.tolist())

In [ ]:
print(df_encoded.columns.tolist())

In [ ]:
X = df_encoded.drop('Customer_Status', axis=1)
y = df_encoded['Customer_Status']

In [ ]:
print(X.select_dtypes(include=['object']).columns)

In [ ]:
print(y.dtype)
print(y.unique())

In [ ]:
X_train.select_dtypes(include=['object']).columns

In [ ]:
y.unique()

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Convert all columns to numeric
X_train = X_train.astype('float64')
X_test = X_test.astype('float64')

# Train model
reg = XGBClassifier()
reg.fit(X_train, y_train)

# Predict
y_predicted = reg.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, y_predicted)

# Plot
plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## Importing Classification Report

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(y_test, y_predicted))

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_predicted)

In the end we conclude that the Telecom Customer Churn Prediction was best worked with XGB_Classifier with an accuracy score of 80.86%